# Task #4: Gate 1 Verification & Reproducibility

In [33]:
import pandas as pd
import numpy as np
import hashlib
import json
from pathlib import Path


## Load
Loaded the split maps from Task #3

In [34]:
# pointing to parent dir
DATA_DIR = "../split_data"           # same folder frozen_train.ipynb reads from
SPLIT_DIR = "../frozen-split_data"   # where frozen_train.ipynb wrote the split map

# Load all 5 normalized dataset tables from Parquet files into pandas DataFrames
contracts = pd.read_parquet(f"{DATA_DIR}/contracts.parquet")
documents = pd.read_parquet(f"{DATA_DIR}/documents.parquet")
categories = pd.read_parquet(f"{DATA_DIR}/categories.parquet")
annotation_sets = pd.read_parquet(f"{DATA_DIR}/annotation_sets.parquet")
spans = pd.read_parquet(f"{DATA_DIR}/spans.parquet")

## Load the frozen 80/20 train/validation split map
split_map = pd.read_parquet(f"{SPLIT_DIR}/frozen-split.parquet")

## Check 1 - Required totals (from Task #1)

The milestone doc requires exactly: 408 Contracts, 408 Documents, 41 Categories, 16,728 Annotation Sets, 11,180 Spans.

In [35]:
required_totals = {
    "contracts": 408,
    "documents": 408,
    "categories": 41,
    "annotation_sets": 16728,
    "spans": 11180,
}
actual_totals = {
    "contracts": len(contracts),
    "documents": len(documents),
    "categories": len(categories),
    "annotation_sets": len(annotation_sets),
    "spans": len(spans),
}
# compare expected vs actual count for each tables
count_checks = []
for key, expected in required_totals.items():
    actual = actual_totals[key]
    count_checks.append({
        "check": f"count_{key}", "expected": expected, "actual": actual,
        "passed": actual == expected,
    })

count_checks_df = pd.DataFrame(count_checks)
count_checks_df

,check,expected,actual,passed
0,count_contracts,408,408,True
1,count_documents,408,408,True
2,count_categories,41,41,True
3,count_annotation_sets,16728,16728,True
4,count_spans,11180,11180,True


## Check 2 - Split isolation & leakage protection

No `context_group_id` should ever appear on both sides of the split, and every document should have gotten a split label in the first place.

In [36]:
# set of context group IDs assigned to train and validate

train_groups = set(split_map.loc[split_map["split"] == "train", "context_group_id"])
val_groups = set(split_map.loc[split_map["split"] == "validation", "context_group_id"])
overlap = train_groups & val_groups

# Left-join documents with split map to verify that each document has a split label
documents_with_split = documents.merge(split_map, on="context_group_id", how="left")

# count documents that are missing a split label (this should be 0)
unassigned = int(documents_with_split["split"].isna().sum())

split_checks = [
    {"check": "train_validation_group_overlap", "expected": 0, "actual": len(overlap), "passed": len(overlap) == 0},
    {"check": "documents_missing_split_label", "expected": 0, "actual": unassigned, "passed": unassigned == 0},
]
split_checks_df = pd.DataFrame(split_checks)

print(split_map["split"].value_counts())
print(split_map["split"].value_counts(normalize=True).round(3))
split_checks_df

split
train         324
validation     83
Name: count, dtype: int64
split
train         0.796
validation    0.204
Name: proportion, dtype: float64


,check,expected,actual,passed
0,train_validation_group_overlap,0,0,True
1,documents_missing_split_label,0,0,True


## Check 3 - Foreign Keys

Every document should point to a real contract; every annotation set to a real contract + category; every span to a real annotation set.
A nonzero count here means something got dropped or mismatched somewhere upstream.

In [37]:
# verify foreign keys to connect across all tables without orphaned records
ref_checks = [
    {"check": "documents_orphaned_contract_id", # verifying each documents links to a valid contract_id
     "actual": int((~documents["contract_id"].isin(contracts["contract_id"])).sum())},
    {"check": "annotation_sets_orphaned_contract_id", # verifying each documents links to a valid contract_id
     "actual": int((~annotation_sets["contract_id"].isin(contracts["contract_id"])).sum())},
    {"check": "annotation_sets_orphaned_category_id", # verifying each documents links to a valid category_id
     "actual": int((~annotation_sets["category_id"].isin(categories["category_id"])).sum())},
    {"check": "spans_orphaned_annotation_set_id", # verifying each documents links to a valid annotation_set_id
     "actual": int((~spans["annotation_set_id"].isin(annotation_sets["annotation_set_id"])).sum())},
]
# all orphan checks are set to 0
for c in ref_checks:
    c["expected"] = 0
    c["passed"] = c["actual"] == 0

ref_checks_df = pd.DataFrame(ref_checks)
ref_checks_df

,check,actual,expected,passed
0,documents_orphaned_contract_id,0,0,True
1,annotation_sets_orphaned_contract_id,0,0,True
2,annotation_sets_orphaned_category_id,0,0,True
3,spans_orphaned_annotation_set_id,0,0,True


## Check 4 - Duplicate keys

Every ID column should be unique within its own table.

In [38]:
# verify primary key uniqueness for each table
dup_checks = [
    {"check": "duplicate_contract_id", "actual": int(contracts["contract_id"].duplicated().sum())},
    {"check": "duplicate_document_id", "actual": int(documents["document_id"].duplicated().sum())},
    {"check": "duplicate_annotation_set_id", "actual": int(annotation_sets["annotation_set_id"].duplicated().sum())},
    {"check": "duplicate_span_id", "actual": int(spans["span_id"].duplicated().sum())},
]
for c in dup_checks:
    c["expected"] = 0
    c["passed"] = c["actual"] == 0

dup_checks_df = pd.DataFrame(dup_checks)
dup_checks_df

,check,actual,expected,passed
0,duplicate_contract_id,0,0,True
1,duplicate_document_id,0,0,True
2,duplicate_annotation_set_id,0,0,True
3,duplicate_span_id,0,0,True


## Check 5 -- Artifact integrity (file hashes)

We hash every parquet file. If a teammate reruns the pipeline from scratch and gets
the exact same hashes, that's proof the pipeline is deterministic and nobody
hand-edited a file after the fact.

In [39]:
# importing inside the cell to prevent nameError if executed out-of-order
def file_md5(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

artifact_files = [
    f"{DATA_DIR}/contracts.parquet",
    f"{DATA_DIR}/documents.parquet",
    f"{DATA_DIR}/categories.parquet",
    f"{DATA_DIR}/annotation_sets.parquet",
    f"{DATA_DIR}/spans.parquet",
    f"{SPLIT_DIR}/frozen-split.parquet",
]
# compute hash dictionary for exisiting files
artifact_hashes = {f: file_md5(f) for f in artifact_files if Path(f).exists()}
missing_artifacts = [f for f in artifact_files if not Path(f).exists()] # track files that are not on the disk

print("missing artifacts (should be empty):", missing_artifacts)
artifact_hashes

missing artifacts (should be empty): []


{'../split_data/contracts.parquet': '58da80fa5d456c26597f50de0955d12d',
 '../split_data/documents.parquet': 'ceb83c1998474d9ebae24c16496eaf7d',
 '../split_data/categories.parquet': '61262c226b2abe37008df56899fa1870',
 '../split_data/annotation_sets.parquet': '7bc833f328d855bfb444990253db942c',
 '../split_data/spans.parquet': '0f47145d141e175af42b8368dd6eda43',
 '../frozen-split_data/frozen-split.parquet': 'c312b2e1656e10ddbab0a0c361d2af0a'}

## Final Gate 1 Report & JSON Export

Everything above collapses into one pass/fail JSON file.

In [40]:
# load task#3 anomaly log (pointing one dir up to notebooks/task3_outputs)
anomaly_log = pd.read_csv("../task3_outputs/anomaly_log.csv")

# filter out non-blocking anomalies
blocking_mask = (anomaly_log["count"] > 0) & (anomaly_log["check"] != "category_low_support_under_5")
blocking_anomalies = anomaly_log[blocking_mask]

blocking_anomalies

,check,count,detail


In [41]:
# all check results combines into one dataframe
all_checks = pd.concat([
    count_checks_df.assign(section="counts"),
    split_checks_df.assign(section="split"),
    ref_checks_df.assign(section="referential_integrity"),
    dup_checks_df.assign(section="duplicates"),
], ignore_index=True)

# pass eval
gate1_passed = bool(all_checks["passed"].all() and blocking_anomalies.empty and not missing_artifacts)

# final report dictionary
gate1_report = {
    "gate1_passed": gate1_passed,
    "checks": all_checks.to_dict(orient="records"),
    "blocking_anomalies": blocking_anomalies.to_dict(orient="records"),
    "artifact_hashes": artifact_hashes,
    "missing_artifacts": missing_artifacts,
}

Path("task4_outputs").mkdir(exist_ok=True)
with open("task4_outputs/gate1_evidence.json", "w") as f:
    json.dump(gate1_report, f, indent=2)

print(f"GATE 1: {'PASSED' if gate1_passed else 'FAILED'}")
all_checks

GATE 1: PASSED


,check,expected,actual,passed,section
0,count_contracts,408,408,True,counts
1,count_documents,408,408,True,counts
2,count_categories,41,41,True,counts
3,count_annotation_sets,16728,16728,True,counts
4,count_spans,11180,11180,True,counts
5,train_validation_group_overlap,0,0,True,split
6,documents_missing_split_label,0,0,True,split
7,documents_orphaned_contract_id,0,0,True,referential_integrity
8,annotation_sets_orphaned_contract_id,0,0,True,referential_integrity
9,annotation_sets_orphaned_category_id,0,0,True,referential_integrity
